# OS-VDKL trên Kaggle — GPT-2 1.5B teacher → 0.1B student

Notebook này chạy distillation **OS-VDKL** trong repo [OS_VDKL](https://github.com/Sonspeed/OS_VDKL).

Trước khi Run All:

1. **Settings → Accelerator → GPU** (T4 / P100 / L4 / A100 đều được).
2. **Settings → Internet → On** (để cài package và tải Hugging Face).
3. Push code OS-VDKL lên branch `OS_VDKL` của `https://github.com/Sonspeed/OS_VDKL` trước khi clone. Notebook chỉ train được đúng code đang có trên GitHub.

Preset mặc định dành cho **T4 16GB**:

- student `MiniLLM/init-gpt2-120M`, teacher `MiniLLM/teacher-gpt2-1.5B`
- data Dolly đã tokenize (`MiniLLM/dolly-processed`, khoảng 15MB)
- 1 GPU, batch 1, grad accumulation 8, seq 512, fp16 (T4/P100 không có bf16)
- không dùng OpenWebText (bản processed khoảng 18GB) và không generate lúc eval, để một session Kaggle chạy hết

Checkpoint ghi vào `/kaggle/working/OS_VDKL/results/...`. Output của notebook giữ các file trong `/kaggle/working`.


## 1. Cấu hình
Sửa ô này rồi chạy từ trên xuống. `PRESET="t4"` tự chọn batch an toàn; đặt `PRESET="custom"` để dùng các số bên dưới.

In [ ]:
import os

# Repo chứa finetune.py và os_vdkl/. Đổi branch nếu code chưa nằm trên main.
REPO_URL = "https://github.com/Sonspeed/OS_VDKL"
GIT_BRANCH = "OS_VDKL"

PRESET = "t4"          # "t4" | "24gb" | "a100" | "custom"
USE_LM = False         # True: thêm OpenWebText LM loss (~18GB download)
USE_INIT_STUDENT = True  # True: MiniLLM/init-gpt2-120M. False: gpt2 gốc (setting nolm)
EVAL_GEN = False       # True: đo ROUGE lúc validate (chậm hơn nhiều)
EPOCHS = 1
MAX_STEPS = 0          # >0 thì dừng sớm, bất kể EPOCHS. 0 = chạy hết epoch
DEV_NUM = 200
SAVE_EVERY = 200
EVAL_EVERY = 200
SEED = 10

# Chỉ dùng khi PRESET == "custom"
BATCH_SIZE = 1
GRAD_ACC = 8
MAX_LENGTH = 512
LR = 1e-4

assert PRESET in {"t4", "24gb", "a100", "custom"}
print("preset", PRESET, "| lm", USE_LM, "| init student", USE_INIT_STUDENT, "| epochs", EPOCHS)


## 2. GPU và package
Giữ PyTorch có sẵn của Kaggle. DeepSpeed được cài không biên dịch CUDA ops (`DS_BUILD_OPS=0`); ZeRO stage 1 của script không cần custom kernel.

In [ ]:
import os, subprocess, sys, shutil

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Vào Settings → Accelerator → GPU, rồi chạy lại từ đầu.")

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
bf16_ok = props.major >= 8
print(f"GPU: {props.name} | {vram_gb:.1f} GB | bf16: {bf16_ok} | torch {torch.__version__}")

if PRESET == "t4":
    BATCH_SIZE, GRAD_ACC, MAX_LENGTH, LR = 1, 8, 512, 1e-4
elif PRESET == "24gb":
    BATCH_SIZE, GRAD_ACC, MAX_LENGTH, LR = 2, 8, 512, 1e-4
elif PRESET == "a100":
    BATCH_SIZE, GRAD_ACC, MAX_LENGTH, LR = 4, 8, 512, 1e-4

if vram_gb < 20 and BATCH_SIZE > 1:
    print(f"VRAM {vram_gb:.1f} GB: hạ batch {BATCH_SIZE} → 1")
    BATCH_SIZE = 1

EVAL_BATCH = 2 if vram_gb < 20 else 4
print(f"train batch {BATCH_SIZE} x grad_acc {GRAD_ACC} | seq {MAX_LENGTH} | lr {LR}")

env = os.environ.copy()
env["DS_BUILD_OPS"] = "0"
env["DS_SKIP_CUDA_CHECK"] = "1"
env["DS_BUILD_AIO"] = "0"
env["DS_BUILD_SPARSE_ATTN"] = "0"

req = [
    "transformers==5.5.0",
    "tokenizers==0.22.2",
    "accelerate==1.13.0",
    "datasets==4.8.4",
    "peft==0.18.1",
    "huggingface_hub==1.9.0",
    "deepspeed==0.18.9",
    "nltk==3.9.4",
    "rouge-score==0.1.2",
    "sentencepiece==0.2.1",
    "protobuf==6.33.6",
    "jsonlines==4.0.0",
    "tqdm==4.67.3",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *req],
    env=env,
)
import torch as torch_after
print("torch after install", torch_after.__version__, "cuda", torch_after.cuda.is_available())
if not torch_after.cuda.is_available():
    raise RuntimeError("pip đã làm mất CUDA của torch. Restart kernel và cài lại, đừng upgrade torch.")


## 3. Lấy code
Trên Kaggle, code được clone vào `/kaggle/working/OS_VDKL` (ghi được). Nếu chạy notebook ngay trong repo local thì dùng thư mục hiện tại.

In [ ]:
import os, subprocess, sys
from pathlib import Path

ON_KAGGLE = os.path.isdir("/kaggle/working")
if ON_KAGGLE:
    ROOT = Path("/kaggle/working/OS_VDKL")
    if not (ROOT / "finetune.py").exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPO_URL, str(ROOT)
        ])
else:
    here = Path.cwd().resolve()
    ROOT = here if (here / "finetune.py").exists() else here.parent if (here.parent / "finetune.py").exists() else here
    if not (ROOT / "finetune.py").exists():
        ROOT = Path.cwd() / "OS_VDKL"
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPO_URL, str(ROOT)
        ])

need = ["finetune.py", "os_vdkl/multi_lora.py", "distillm/losses.py", "configs/deepspeed/ds_config.json"]
missing = [p for p in need if not (ROOT / p).exists()]
if missing:
    raise FileNotFoundError(
        "Repo thiếu file OS-VDKL: " + ", ".join(missing)
        + ". Hãy push code lên GitHub rồi chạy lại ô này (xoá /kaggle/working/OS_VDKL nếu clone cũ)."
    )
os.chdir(ROOT)
print("ROOT", ROOT)


## 4. Tải student, teacher và Dolly

- Student init: `MiniLLM/init-gpt2-120M` (khoảng 0.5GB)
- Teacher SFT: `MiniLLM/teacher-gpt2-1.5B` (khoảng 6GB, 2 shard)
- Dolly tokenized: `MiniLLM/dolly-processed` nhánh `full/gpt2`

`USE_LM=True` tải thêm OpenWebText processed (~18GB) vào `processed_data/openwebtext/gpt2/512/10M/`.


In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

os.environ.setdefault("HF_HOME", str(ROOT / ".hf_home"))
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

if USE_INIT_STUDENT:
    student_dir = ROOT / "results/gpt2/train/init/gpt2-base"
    student_repo = "MiniLLM/init-gpt2-120M"
else:
    student_dir = ROOT / "checkpoints/gpt2-base"
    student_repo = "gpt2"

teacher_dir = ROOT / "results/gpt2/train/sft/gpt2-xlarge"
data_dir = ROOT / "processed_data/dolly/full/gpt2"
lm_dir = ROOT / "processed_data/openwebtext/gpt2/512/10M"

def grab(repo, dest, repo_type="model", allow_patterns=None):
    dest.mkdir(parents=True, exist_ok=True)
    marker = dest / "config.json" if repo_type == "model" else dest / "train_0.idx"
    if marker.exists():
        print("đã có", dest)
        return
    print("tải", repo, "→", dest)
    snapshot_download(
        repo,
        repo_type=repo_type,
        local_dir=str(dest),
        allow_patterns=allow_patterns,
    )

grab(student_repo, student_dir)
grab("MiniLLM/teacher-gpt2-1.5B", teacher_dir)
grab(
    "MiniLLM/dolly-processed",
    ROOT / "processed_data/dolly",
    repo_type="dataset",
    allow_patterns=["full/gpt2/*"],
)
if not (data_dir / "train_0.idx").exists():
    raise FileNotFoundError(f"Không thấy Dolly indexed data tại {data_dir}")

if USE_LM:
    grab(
        "MiniLLM/openwebtext-processed",
        lm_dir,
        repo_type="dataset",
        allow_patterns=["train_0.bin", "train_0.idx", "valid_0.bin", "valid_0.idx"],
    )

print("student", student_dir)
print("teacher", teacher_dir)
print("data", data_dir)


## 5. Train

Một GPU. T4/P100 dùng `configs/deepspeed/ds_config.json` (fp16). GPU Ampere trở lên (A100, L4, RTX 30/40) dùng config bf16 của paper.

Loss OS-VDKL giữ nguyên: 3 nhánh LoRA rank 16, top-64, Sinkhorn 5 bước, `student-temp 2`, `kd-ratio 1.0`.

Nếu CUDA out of memory: đặt `PRESET="custom"`, `BATCH_SIZE=1`, `MAX_LENGTH=256`, rồi chạy lại từ ô cấu hình. Dolly khoảng 14k mẫu; batch 1 × grad acc 8 là khoảng 1750 optimizer step mỗi epoch, thường 1–2.5 giờ trên T4, nên `EPOCHS=1` là mặc định.


In [ ]:
import os, subprocess, sys
from pathlib import Path

import torch

bf16_ok = torch.cuda.get_device_properties(0).major >= 8
ds_config = ROOT / ("configs/deepspeed/ds_config_zero1_bf16.json" if bf16_ok else "configs/deepspeed/ds_config.json")
save_path = ROOT / "results/gpt2/train/kaggle_os_vdkl"
save_path.mkdir(parents=True, exist_ok=True)

cmd = [
    "torchrun", "--nproc_per_node", "1", "--master_port", "2012", "finetune.py",
    "--base-path", str(ROOT),
    "--model-path", str(student_dir),
    "--teacher-model-path", str(teacher_dir),
    "--ckpt-name", "gpt2-base",
    "--teacher-ckpt-name", "xlarge-sft",
    "--teacher-model-fp16",
    "--n-gpu", "1",
    "--model-type", "gpt2",
    "--data-dir", str(data_dir) + "/",
    "--num-workers", "2",
    "--dev-num", str(DEV_NUM),
    "--lr", str(LR),
    "--batch-size", str(BATCH_SIZE),
    "--eval-batch-size", str(EVAL_BATCH),
    "--gradient-accumulation-steps", str(GRAD_ACC),
    "--warmup-iters", "0",
    "--lr-decay-style", "cosine",
    "--weight-decay", "1e-2",
    "--clip-grad", "1.0",
    "--epochs", str(EPOCHS),
    "--kd-ratio", "1.0",
    "--max-length", str(MAX_LENGTH),
    "--max-prompt-length", "256",
    "--do-train",
    "--do-valid",
    "--save-interval", str(SAVE_EVERY),
    "--eval-interval", str(EVAL_EVERY),
    "--log-interval", "10",
    "--mid-log-num", "-1",
    "--save", str(save_path),
    "--seed", str(SEED),
    "--deepspeed",
    "--deepspeed_config", str(ds_config),
    "--type", "adaptive-os-vdkl" if USE_LM else "os-vdkl",
    "--do-sample",
    "--top-k", "0",
    "--top-p", "1.0",
    "--temperature", "1.0",
    "--os-vdkl",
    "--os-num-branches", "3",
    "--os-top-m", "64",
    "--os-lora-rank", "16",
    "--os-lora-alpha", "16.0",
    "--os-init-scale", "0.02",
    "--os-sinkhorn-eps", "0.05",
    "--os-sinkhorn-iters", "5",
    "--os-lambda-local", "0.5",
    "--os-mu-ortho", "0.1",
    "--os-eta-sem", "0.05",
    "--os-sgo-mode", "random_branch",
    "--os-eval-mode", "all",
    "--student-temp", "2",
]
if USE_LM:
    cmd += [
        "--lm-data-dir", str(lm_dir) + "/",
        "--student-gen",
        "--gen-num-beams", "1",
        "--gen-top-p", "1.0",
        "--init-threshold", "0.0",
        "--loss-eps", "0.1",
        "--capacity", "1000",
    ]
if EVAL_GEN:
    cmd.append("--eval-gen")
if MAX_STEPS and int(MAX_STEPS) > 0:
    cmd += ["--total-iters", str(int(MAX_STEPS))]

env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT)
env["WANDB_DISABLED"] = "true"
env["TF_CPP_MIN_LOG_LEVEL"] = "3"
env["TOKENIZERS_PARALLELISM"] = "false"
env["NCCL_DEBUG"] = "WARN"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["OS_VDKL_PROGRESS"] = "1"
env["PYTHONUNBUFFERED"] = "1"

print("deepspeed config:", ds_config.name)
print("save:", save_path)
print(" ".join(cmd))
proc = subprocess.Popen(
    cmd, cwd=str(ROOT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
from tqdm.auto import tqdm
bar = None
log_file = save_path / "kaggle_stdout.log"
with log_file.open("w") as handle:
    for line in proc.stdout:
        handle.write(line)
        if line.startswith("@@progress_total "):
            total = int(line.split()[1])
            bar = tqdm(total=total, desc="train", unit="step")
            continue
        if line.startswith("@@progress "):
            _, done, total, loss, ds_loss, lr, secs = line.split()
            if bar is None:
                bar = tqdm(total=int(total), desc="train", unit="step")
            done = int(done)
            if done > bar.n:
                bar.update(done - bar.n)
            bar.set_postfix(loss=loss, ds=ds_loss, lr=lr, sec=secs)
            continue
        sys.stdout.write(line)
if bar is not None:
    bar.close()
code = proc.wait()
print("exit", code, "| stdout copy:", log_file)
if code != 0:
    raise SystemExit(code)


## 6. Checkpoint
Mỗi lần save gồm weight Hugging Face và `os_multi_lora.pt` (3 nhánh LoRA). File này là thứ cần có khi eval OS-VDKL.

In [ ]:
from pathlib import Path

logs = sorted(save_path.glob("**/log.txt"))
print("logs:", logs)
if logs:
    text = logs[-1].read_text(errors="replace").splitlines()
    print("\n".join(text[-30:]))

ckpts = sorted(
    [p.parent for p in save_path.glob("**/os_multi_lora.pt")],
    key=lambda p: p.stat().st_mtime,
)
print("\ncheckpoints:")
for p in ckpts:
    print(" -", p)
if not ckpts:
    print("Chưa có os_multi_lora.pt. Train chưa tới save-interval, hoặc bước train bị dừng trước đó.")


## 7. Đẩy checkpoint lên Hugging Face
Ô này gửi toàn bộ những gì đã lưu trong `save_path`: log, từng checkpoint theo step, và `final/` (kèm `os_multi_lora.pt`). Thêm secret `HF_TOKEN` trong Kaggle, rồi điền `HF_REPO`.

In [ ]:
import os
from pathlib import Path

# Điền repo của bạn. Ví dụ "ten-ban/os-vdkl-gpt2-0.1b".
# Token: Kaggle → Add-ons → Secrets → tên HF_TOKEN, hoặc gán thẳng vào HF_TOKEN.
HF_REPO = ""
HF_PRIVATE = True
HF_TOKEN = ""

def resolve_token():
    token = (HF_TOKEN or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or "").strip()
    if token:
        return token
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret("HF_TOKEN").strip()

if not str(HF_REPO).strip() or "/" not in HF_REPO:
    raise ValueError('Đặt HF_REPO = "username/ten-repo" trước khi chạy ô này.')

root = Path(save_path)
if not root.is_dir():
    raise FileNotFoundError(f"Chưa có thư mục checkpoint: {root}")

saved = [p for p in root.rglob("*") if p.is_file() and p.stat().st_size > 0]
if not saved:
    raise FileNotFoundError(f"{root} trống, chưa có gì để đẩy.")

card = root / "README.md"
if not card.exists():
    card.write_text(
        "---\n"
        "license: mit\n"
        "base_model: MiniLLM/init-gpt2-120M\n"
        "tags:\n"
        "- os-vdkl\n"
        "- knowledge-distillation\n"
        "- gpt2\n"
        "---\n\n"
        "# OS-VDKL GPT-2 0.1B\n\n"
        "Checkpoint từ notebook Kaggle. Mỗi thư mục số bước và `final/` gồm weight student "
        "cùng `os_multi_lora.pt` (3 nhánh LoRA). `AutoModelForCausalLM` chỉ nạp weight student; "
        "gắn nhánh bằng `os_vdkl.multi_lora.attach_os_multi_lora`.\n"
    )

token = resolve_token()
from huggingface_hub import HfApi

api = HfApi(token=token)
api.create_repo(HF_REPO, exist_ok=True, private=HF_PRIVATE, repo_type="model")
print(f"đẩy {len(saved)} file từ {root}")
for path in sorted(saved):
    print(f"  {path.relative_to(root)}  {path.stat().st_size / 1e6:.1f} MB")
commit = api.upload_folder(
    folder_path=str(root),
    repo_id=HF_REPO,
    repo_type="model",
    commit_message="Upload OS-VDKL Kaggle checkpoints",
)
print("xong", f"https://huggingface.co/{HF_REPO}")
print(commit)
